## Step 1: Create role: iceberg_management_role

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE ROLE iceberg_management_role;

GRANT CREATE DATABASE, 
      CREATE EXTERNAL VOLUME
ON ACCOUNT 
TO ROLE iceberg_management_role;



## Step2: Create database and schema: iceberg_demo_db.iceberg_schema

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE DATABASE iceberg_demo_db;
CREATE OR REPLACE SCHEMA iceberg_schema;

## Step 3: Grant USAGE privileges to the iceberg_management_role role 

In [ ]:
%%sql -r dataframe_3
GRANT CREATE ICEBERG TABLE ON SCHEMA iceberg_demo_db.iceberg_schema TO ROLE iceberg_management_role;

GRANT USAGE ON DATABASE iceberg_demo_db TO ROLE iceberg_management_role;
GRANT USAGE ON SCHEMA iceberg_demo_db.iceberg_schema TO ROLE iceberg_management_role;

## Step 4: Create a new S3 bucket
### Switch to AWS and create a new S3 bucket, and also capture AWS account id to use in the next SQL command:

- navigate to https://aws.amazon.com/console/
- Account ID: aws4-studentX
- IAM username: student
- password: ...

- Once connected find your AWS Account ID (595239850147)
- Create New S3 bucket in the same region that you use for your Snowflake Account (S3 Bucket Name = iceberg-storage-zahar-h)


## Step 5: Create external volume object (iceberg_external_volume)

In [ ]:
aws_account_id = 595239850147
s3_bucket      = 'iceberg-storage-zahar-h'
role_name      = 'SnowflakeIcebergAccessRole'
ext_volume     = 'ICEBERG_EXTERNAL_VOLUME'
policy_name    = 'InlineSnowflakeIcebergS3'

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

sql = f"""
    CREATE OR REPLACE EXTERNAL VOLUME {ext_volume}
    STORAGE_LOCATIONS = (
        (
          NAME = 'default_s3_loc'
          STORAGE_PROVIDER = 'S3'
          STORAGE_BASE_URL = 's3://{s3_bucket}/'
          STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::{aws_account_id}:role/{role_name}'
        )
    )
    ALLOW_WRITES = TRUE
    COMMENT = 'Volume for Iceberg tables in {s3_bucket}'
    """

print(sql)

session.sql(sql).collect()  # execute the DDL
print(f"EXTERNAL VOLUME {ext_volume} has been created")

## Step 6: Describe the EXTERNAL VOLUME object

to get 
- Snowflake AWS Account ID (from STORAGE_AWS_IAM_USER_ARN)
- STORAGE_AWS_EXTERNAL_ID

In [ ]:
%%sql -r dataframe_4
DESC EXTERNAL VOLUME ICEBERG_EXTERNAL_VOLUME
->> SELECT "property_value" FROM $1 WHERE "property"='STORAGE_LOCATION_1'

In [ ]:
DESC EXTERNAL VOLUME ICEBERG_EXTERNAL_VOLUME
->>
SELECT
    PARSE_JSON("property_value") j
FROM $1
WHERE "property" = 'STORAGE_LOCATION_1'
->>
SELECT
  SPLIT_PART(SPLIT_PART(j:STORAGE_AWS_IAM_USER_ARN::STRING, '::', 2), ':', 1) AS snowflake_aws_account_id,
  j:STORAGE_AWS_EXTERNAL_ID::STRING AS external_id
FROM $1;

In [ ]:
_sql = """
DESC EXTERNAL VOLUME ICEBERG_EXTERNAL_VOLUME
->>
SELECT
    PARSE_JSON("property_value") j
FROM $1
WHERE "property" = 'STORAGE_LOCATION_1'
->>
SELECT
  SPLIT_PART(SPLIT_PART(j:STORAGE_AWS_IAM_USER_ARN::STRING, '::', 2), ':', 1) AS snowflake_aws_account_id,
  j:STORAGE_AWS_EXTERNAL_ID::STRING AS external_id
FROM $1;
"""

row = session.sql(_sql).collect()[0]

sf_acct = row["SNOWFLAKE_AWS_ACCOUNT_ID"]
external_id  = row["EXTERNAL_ID"]

print("Snowflake AWS Account ID:", sf_acct)
print("External ID:", external_id)

## Step 7: Take Snowflake AWS Account ID and External ID values 

and use them for AIM Role Trust Policy (in AWS):
- create SnowflakeIcebergAccessRole with InlineSnowflakeIcebergS3 policy

In [ ]:
import json

# ---- 1) Trust policy JSON ----
trust_doc = {
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": { "AWS": f"arn:aws:iam::{sf_acct}:root" },
      "Action": "sts:AssumeRole",
      "Condition": {
        "StringEquals": {
          "sts:ExternalId": external_id
        }
      }
    }
  ]
}

# ---- 2) Inline S3 permissions policy JSON (scoped to your bucket) ----
s3_policy = {
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "BucketLevel",
      "Effect": "Allow",
      "Action": ["s3:ListBucket", "s3:GetBucketLocation"],
      "Resource": f"arn:aws:s3:::{s3_bucket}"
    },
    {
      "Sid": "ObjectLevel",
      "Effect": "Allow",
      "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject", "s3:DeleteObjectVersion"],
      "Resource": f"arn:aws:s3:::{s3_bucket}/*"
    }
  ]
}

print(f"=== Create Role: {role_name}===")
print("=== Trust policy ===")
print(json.dumps(trust_doc, indent=2))
print(f"\n=== Inline S3 policy: {policy_name}===")
print(json.dumps(s3_policy, indent=2))


### in IAM: 

- go to Roles
- Click 'Create Role'
- Use Choose Custom trust policy
- Copy and paste Trust policy from the above step and click "Next"
- Skip "Add permissions" by clicking "Next"
- Enter Role Name: SnowflakeIcebergAccessRole and click "Create Role"
- Find newly created role in Roles and click on it
- Under Add permission drop-down menu, click on "Create inline policy"
- Switch to JSON view and paste the above Inline S3 policy json into the Policy Editor, click "Next"
- Enter Policy name: InlineSnowflakeIcebergS3 and click "Create Policy"




## Step 8: Create Iceberg Table 

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE ICEBERG TABLE iceberg_demo_db.iceberg_schema.orders (
  order_id INT,
  customer_id INT,
  order_date DATE,
  total_amount NUMBER(10,2)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'ICEBERG_EXTERNAL_VOLUME'
BASE_LOCATION = 'orders/';   -- optional, to control subdirectory under volume

## Step 9: Insert rows

In [ ]:
%%sql -r dataframe_6
INSERT INTO iceberg_demo_db.iceberg_schema.orders
  (order_id, customer_id, order_date, total_amount)
VALUES
  (1, 101, '2025-09-17', 150.75),
  (2, 102, '2025-09-18', 200.00);

## Step 10: Verify in S3

Open the bucket iceberg-storage-zahar-h and look for a new folder like:

orders.<random_suffix>/

&nbsp;&nbsp;&nbsp;metadata/   -- Iceberg metadata & snapshots (JSON/Avro)

&nbsp;&nbsp;&nbsp;data/       -- Parquet files with your rows

#### (Optional) Prove updates/deletes create new snapshots:

In [ ]:
%%sql -r dataframe_7
UPDATE iceberg_demo_db.iceberg_schema.orders
SET total_amount = 175.00
WHERE order_id = 1;

DELETE FROM iceberg_demo_db.iceberg_schema.orders
WHERE order_id = 2;

#### Refresh S3 (data/): you’ll see additional snapshot/manifest files; old Parquet files aren’t overwritten.

In [ ]:
-- Explanation on where each identifier is used:

/* 
Bucket name: iceberg-storage-zahar-h	
Where it goes: In your S3 policy 

Resource ARNs: 
arn:aws:s3:::iceberg-storage-zahar-h 
and 
arn:aws:s3:::iceberg-storage-zahar-h/*

-------------------------------------------

Your AWS account ID: 595239850147	
Where it goes: In CREATE EXTERNAL VOLUME → 
STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::595239850147:role/SnowflakeIcebergAccessRole'

-------------------------------------------
Snowflake AWS account ID: 837042614963	
Where it goes: In IAM trust policy → 
"Principal": { "AWS": "arn:aws:iam::837042614963:root" }

-------------------------------------------
ExternalId: GCC38978_SFCRole=4_X5GBN80b6u0QI2wy1XbalSBfPgI=	
Where it goes: In IAM trust policy → 
"Condition": { "StringEquals": { "sts:ExternalId": "<value>" } }

*/

In [ ]:
%%sql -r dataframe_9
SELECT *
FROM orders

## Step 11: Time Travel with Iceberg Tables:

In [ ]:
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN TABLE orders;

### Check the table as of 3 min ago

In [ ]:
SELECT *
FROM orders AT (OFFSET => -3*60);

### Check the table as of a specific point in time

In [ ]:
SELECT *
FROM orders AT (
  TIMESTAMP => DATEADD(minute, -4, CURRENT_TIMESTAMP())
);

### Find the last DELETE statement and use its query_id to see the data right before the DELETE was executed:


In [ ]:
--Verify you find the right one:

SELECT query_id, query_text, start_time
FROM TABLE(information_schema.query_history_by_session())
WHERE query_type = 'DELETE'
ORDER BY start_time DESC
LIMIT 1;

In [ ]:
SET id = (SELECT query_id
FROM TABLE(information_schema.query_history_by_session())
WHERE query_type = 'DELETE'
ORDER BY start_time DESC
LIMIT 1)

In [ ]:
SELECT *
FROM orders
BEFORE (STATEMENT => $id);

### Cleanup:

- Drop role, database, external volumn in Snowflake
- Delete S3 bucket, SnowflakeIcebergAccessRole IAM role - in AWS console

In [ ]:

drop role if exists iceberg_management_role;
drop database if exists ICEBERG_DEMO_DB;
drop EXTERNAL VOLUME if exists ICEBERG_EXTERNAL_VOLUME;